In [1]:
import albumentations as A
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import os
import segmentation_models as sm
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import Sequence, to_categorical

from unet import *

print(f'TensorFlow version: {tf.__version__}')

Segmentation Models: using `keras` framework.
TensorFlow version: 2.12.0


In [ ]:
BATCH_SIZE = 4
EPOCHS = 50
STEPS_PER_EPOCH = 5000
LEARNING_RATE = 3e-4
MOMENTUM = 0.5

HEIGHT = 240  # 480
WIDTH = 320  # 640
CHANNELS = 3
IMAGE_SHAPE = (HEIGHT, WIDTH, CHANNELS)

class_weights = np.array([1.0591986, 14.61948805, 15.71746382, 5.09713808, 44.17992558, 1., 4.78133668, 2.42209091])
seed = 411

DEEP_SUPERVISION = False
RGB_MODE = False

In [ ]:
data_path = 'input/'
train_path = os.path.join(data_path, 'train_val')
test_path = os.path.join(data_path, 'test')

train_images_path = os.path.join(train_path, 'images')
train_masks_path = os.path.join(train_path, 'masks')

test_images_path = os.path.join(test_path, 'images')
test_masks_path = os.path.join(test_path, 'masks')

class_names = [
    'Background (waterbody)', 'Human divers', 'Aquatic plants and sea-grass',
    'Wrecks and ruins', 'Robots (AUVs/ROVs/instruments)', 'Reefs and invertebrates',
    'Fish and vertebrates', 'Sea-floor and rocks',
]
num_classes = len(class_names)
class_codes = {
    0: [0, 0, 0],  # Black
    1: [0, 0, 1],  # Blue
    2: [0, 1, 0],  # Green
    3: [0, 1, 1],  # Sky
    4: [1, 0, 0],  # Red
    5: [1, 0, 1],  # Pink
    6: [1, 1, 0],  # Yellow
    7: [1, 1, 1],  # White
}

for i, name in enumerate(class_names):
    print(f'Class {i}: {name}')

In [ ]:
def list_files(path):
    content = os.listdir(path)
    # Get only files (not directories)
    files = [os.path.join(path, f) for f in content if os.path.isfile(os.path.join(path, f))]
    # Ensure the lists are sorted to match the corresponding image and mask
    files.sort()
    return files


def rgb_to_label(image, label_codes, one_hot=True):
    label_image = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
    # Assigns each pixel a label from [0, num_classes - 1]
    for class_num, color in label_codes.items():
        label_image[np.all(image == color, axis=-1)] = class_num
    # Converts labeled image from (H, W) to (H, W, num_classes),
    # where each pixel is a one-hot vector (e.g. [2] => [0, 0, 1, 0, 0, 0, 0, 0])
    if one_hot:
        n_classes = len(label_codes)
        label_image = to_categorical(label_image, n_classes)
    return label_image


def label_to_rgb(image, label_codes, one_hot=True):
    rgb_image = np.empty((image.shape[0], image.shape[1], 3), dtype=np.float32)
    # Assigns each pixel a color from label_codes
    if one_hot:
        for class_num, color in label_codes.items():
            rgb_image[image[:, :, class_num] == 1] = color
    else:
        for class_num, color in label_codes.items():
            rgb_image[image == class_num] = color
    return rgb_image


def load_and_preprocess(image_path, mask_path, width=None, height=None, augmentation=None,
                        normalize=True, keep_rgb_mask=False, one_hoe_encode=False):
    # Load images
    img = cv.imread(image_path, cv.IMREAD_COLOR)
    mask = cv.imread(mask_path, cv.IMREAD_COLOR)

    # Convert to RGB
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    mask = cv.cvtColor(mask, cv.COLOR_BGR2RGB)

    # Resize
    if width is not None and height is not None:
        img = cv.resize(img, (width, height), interpolation=cv.INTER_NEAREST)
        mask = cv.resize(mask, (width, height), interpolation=cv.INTER_NEAREST)

    # Augment
    if augmentation is not None:
        augmented = augmentation(image=img, mask=mask)
        img = augmented['image']
        mask = augmented['mask']

    # Fix mask values to be 0 or 255
    if not np.all(np.logical_or(mask == 0, mask == 255)):
        mask = np.round(mask / 255.0) * 255.0

    # Normalize
    if normalize:
        img = (img / 255.0).astype(np.float32)
        mask = (mask / 255.0).astype(np.float32)

    # Return RGB image and mask
    if keep_rgb_mask:
        return img, mask

    # Convert mask to:
    # - label encoded image (each pixel is an integer from 0 to num_classes - 1)
    # - one-hot encoded image (each pixel is a one-hot vector with 1 in the class index)
    mask = rgb_to_label(mask, class_codes, one_hot=one_hoe_encode)
    return img, mask


def iou_coef_2(y_true, y_pred, smooth=1):
    intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
    union = K.sum(y_true, [1, 2, 3]) + K.sum(y_pred, [1, 2, 3]) - intersection
    iou = K.mean((intersection + smooth) / (union + smooth), axis=0)
    return iou


def dice_coef_2(y_true, y_pred, smooth=1):
    intersection = K.sum(y_true * y_pred, axis=[1, 2, 3])
    union = K.sum(y_true, axis=[1, 2, 3]) + K.sum(y_pred, axis=[1, 2, 3])
    dice = K.mean((2. * intersection + smooth) / (union + smooth), axis=0)
    return dice


# IoU = Jaccard index
def jaccard_coef(y_true, y_pred, smooth: float = 1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(K.abs(y_true_f * y_pred_f))
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return K.mean((intersection + smooth) / (union + smooth))


# f1 score = Dice coefficient
def dice_coef(y_true, y_pred, smooth: float = 1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    area_true = K.sum(y_true_f)
    area_pred = K.sum(y_pred_f)
    return K.mean((2. * intersection + smooth) / (area_true + area_pred + smooth))


def jaccard_loss(y_true, y_pred):
    return 1.0 - jaccard_coef(y_true, y_pred)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def focal_loss(y_true, y_pred, gamma=2., alpha=4., epsilon=1.e-9):
    y_true_c = tf.convert_to_tensor(y_true, tf.float32)
    y_pred_c = tf.convert_to_tensor(y_pred, tf.float32)

    model_out = tf.add(y_pred_c, epsilon)
    ce = tf.multiply(y_true_c, -tf.math.log(model_out))
    weight = tf.multiply(y_true_c, tf.pow(tf.subtract(1., model_out), gamma))
    fl = tf.multiply(alpha, tf.multiply(weight, ce))

    reduced_fl = tf.reduce_max(fl, axis=-1)
    return tf.reduce_mean(reduced_fl)


def hybrid_loss(y_true, y_pred):
    j_loss = jaccard_loss(y_true, y_pred)
    d_loss = dice_loss(y_true, y_pred)
    f_loss = focal_loss(y_true, y_pred)
    return j_loss + d_loss + f_loss


In [ ]:
class DataGenerator(Sequence):

    def __init__(self, image_paths, mask_paths, batch_size, img_height, img_width,
                 augmentation=None, shuffle=True, rgb_mask=True, preprocess_fn=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.img_height = img_height
        self.img_width = img_width
        self.augmentation = augmentation
        self.shuffle = shuffle
        self.rgb_mask = rgb_mask
        self.preprocess_fn = preprocess_fn
        self.on_epoch_end()

    def __len__(self):
        return len(self.image_paths) // self.batch_size

    def __getitem__(self, index):
        indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]

        image_paths_temp = [self.image_paths[k] for k in indexes]
        mask_paths_temp = [self.mask_paths[k] for k in indexes]

        images, masks = self.__data_generation(image_paths_temp, mask_paths_temp)

        return images, masks

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.image_paths))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, image_paths_temp, mask_paths_temp):
        # Initialize batch arrays
        images = []
        masks = []

        for img_path, mask_path in zip(image_paths_temp, mask_paths_temp):
            # Load and preprocess image and mask
            img, mask = load_and_preprocess(
                img_path, mask_path, width=self.img_width, height=self.img_height,
                augmentation=self.augmentation, keep_rgb_mask=self.rgb_mask,
                normalize=True, one_hoe_encode=True,
            )
            if self.preprocess_fn is not None:
                img = self.preprocess_fn(img)
            # Add to batch
            images.append(img)
            masks.append(mask)

        # Apply extra preprocessing on input images
        # if self.preprocess_fn is not None:
        #     images = self.preprocess_fn(images)

        return np.array(images), np.array(masks)


class VisualizeProgress(tf.keras.callbacks.Callback):
    def __init__(self, test_data, every_n_epochs=10, num_samples=4, threshold=None):
        super(VisualizeProgress, self).__init__()
        self.test_data = test_data
        self.every_n_epochs = every_n_epochs
        self.num_samples = num_samples
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n_epochs == 0:
            x_test, y_test = self.test_data
            predictions = self.model.predict(x_test[:self.num_samples])

            if DEEP_SUPERVISION:
                predictions = predictions[-1]

            plt.figure(figsize=(12, 12))
            for index in range(self.num_samples):
                if index >= len(x_test): break

                plt.subplot(self.num_samples, 3, index * 3 + 1)
                plt.imshow(x_test[index])
                if index == 0: plt.title('Input Image')

                plt.subplot(self.num_samples, 3, index * 3 + 2)
                if self.threshold is None:
                    ground_truth = label_to_rgb(y_test[index], class_codes, one_hot=True)
                else:
                    ground_truth = y_test[index]
                plt.imshow(ground_truth)
                if index == 0: plt.title('Ground Truth')

                plt.subplot(self.num_samples, 3, index * 3 + 3)
                if self.threshold is None:
                    pred_argmax = np.argmax(predictions[index], axis=-1)
                    prediction = label_to_rgb(pred_argmax, class_codes, one_hot=False)
                else:
                    prediction = (predictions[index] > self.threshold).astype(np.uint8) * 255
                plt.imshow(prediction)
                if index == 0: plt.title('Predicted Segmentation')

            plt.show()


## Preprocessing

In [ ]:
dataset_dir = 'input/example'
train_images_path = os.path.join(dataset_dir, 'images')
train_masks_path = os.path.join(dataset_dir, 'masks')

# Get image and mask paths
train_val_images = list_files(train_images_path)
train_val_masks = list_files(train_masks_path)
test_images = list_files(test_images_path)
test_masks = list_files(test_masks_path)

# Split into training and validation sets
train_images, val_images, train_masks, val_masks = train_test_split(
    train_val_images, train_val_masks, test_size=0.2, random_state=seed
)

print('Training images:', len(train_images))
print('Validation images:', len(val_images))
print('Test images:', len(test_images))

# Check if image names match with mask names
for i in range(4):
    if i < len(train_images): print(train_images[i], '|', train_masks[i])
    if i < len(val_images): print(val_images[i], '|', val_masks[i])
    if i < len(test_images): print(test_images[i], '|', test_masks[i])

In [ ]:
# Augment data
data_augmentation = A.Compose([
    A.HorizontalFlip(p=0.5),
    # A.VerticalFlip(p=0.5),
    A.Rotate(limit=45, p=0.75, interpolation=cv.INTER_NEAREST, border_mode=cv.BORDER_REFLECT_101),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.RandomGamma(gamma_limit=(80, 120), p=0.5),
    ], p=0.75),
    A.OneOf([
        A.Blur(blur_limit=3, p=0.5),
        A.MedianBlur(blur_limit=3, p=0.5),
    ], p=0.75),
])

# Create data generator
train_generator = DataGenerator(
    train_images, train_masks, BATCH_SIZE, HEIGHT, WIDTH,
    rgb_mask=RGB_MODE, augmentation=data_augmentation,
)
val_generator = DataGenerator(
    val_images, val_masks, BATCH_SIZE, HEIGHT, WIDTH, rgb_mask=RGB_MODE,
)
test_generator = DataGenerator(
    test_images, test_masks, BATCH_SIZE, HEIGHT, WIDTH, rgb_mask=RGB_MODE,
)

print('Number of batches in training set:', len(train_generator))
print('Number of batches in validation set:', len(val_generator))
print('Number of batches in test set:', len(test_generator))

batch = train_generator[0]
images, masks = batch
print('Training image batch shape:', images.shape)
print('Training mask batch shape:', masks.shape)

# Show some iamge-mask pairs for sanity check
for i, (img, mask) in enumerate(zip(images, masks)):
    if i > 4: break
    plt.figure(figsize=(12, 12))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.axis('off')
    plt.subplot(1, 2, 2)
    if RGB_MODE:  # RGB
        plt.imshow(mask)
    else:
        plt.imshow(label_to_rgb(mask, class_codes))
    plt.axis('off')
    plt.show()

## Training

In [ ]:
# SM losses
sm_dice = sm.losses.DiceLoss(class_weights=class_weights)
sm_focal = sm.losses.CategoricalFocalLoss()
sm_cce = sm.losses.CategoricalCELoss(class_weights=class_weights)
sm_jac = sm.losses.JaccardLoss(class_weights=class_weights)
sm_total_loss = (1. * sm_dice) + (3. * sm_focal) + (1. * sm_cce) + (1. * sm_jac)

# SM metrics
sm_iou = sm.metrics.IOUScore(threshold=0.5)
sm_f1 = sm.metrics.FScore(threshold=0.5)

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=MOMENTUM)

model = UNet(
    input_shape=IMAGE_SHAPE,
    num_classes=3 if RGB_MODE else num_classes,  # 3 or num_classes
    num_filters=32,
    dropout=0.,
    batchnorm=True,
    final_activation='sigmoid' if RGB_MODE else 'softmax',  # sigmoid or softmax
)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy' if RGB_MODE else 'categorical_crossentropy',
    metrics=['accuracy', jaccard_coef, dice_coef, sm_iou, sm_f1],
)

# tf.keras.utils.plot_model(model, show_shapes=True)
# model.summary()

In [ ]:
EPOCHS = 3
STEPS_PER_EPOCH = 2

history = model.fit(
    train_generator, validation_data=val_generator, epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH, validation_steps=len(val_generator),
    callbacks=[
        ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, verbose=1),
        EarlyStopping(monitor='val_loss', patience=10, verbose=1),
        VisualizeProgress(val_generator[0], every_n_epochs=1, num_samples=5, threshold=0.5 if RGB_MODE else None),
    ],
    verbose=1,
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(121)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title('Loss')
plt.subplot(122)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.legend()
plt.title('Accuracy')
plt.show()

In [ ]:
# save model
model.save('models/basic-unet-model.h5')

## Evaluation

In [ ]:
# load model
model = tf.keras.models.load_model('models/basic-unet-model.h5', custom_objects={
    'jaccard_coef_loss': lambda y_true, y_pred: -jaccard_coef(y_true, y_pred),
    'jaccard_coef': jaccard_coef,
    'dice_coef': dice_coef,
})

In [ ]:
# load model for testing
model = tf.keras.models.load_model('models/sm-unet-model.h5', compile=False)

In [ ]:
test = test_generator[0]

predictions = model.predict(test[0])
predictions.shape

In [ ]:
y_pred = np.argmax(predictions, axis=-1)

plt.subplots(3, 4, figsize=(12, 7))
for i, pred in enumerate(y_pred[:4]):
    img = test[0][i]
    true = test[1][i]
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.axis('off')
    plt.subplot(3, 4, i + 5)
    if RGB_MODE:
        plt.imshow(true)
    else:
        plt.imshow(label_to_rgb(true, class_codes, one_hot=True))
    plt.axis('off')
    plt.subplot(3, 4, i + 9)
    if RGB_MODE:
        plt.imshow(pred)
    else:
        plt.imshow(label_to_rgb(pred, class_codes, one_hot=False))
    plt.axis('off')
plt.show()

## Transfer Learning with Segmentation Models

In [ ]:
HEIGHT = 256
WIDTH = 256
CHANNELS = 3
IMAGE_SHAPE = (HEIGHT, WIDTH, CHANNELS)
EPOCHS = 2
STEPS_PER_EPOCH = 2

BACKBONE = 'resnet34'
preprocess_input = sm.get_preprocessing(BACKBONE)

# Create data generator
train_generator = DataGenerator(train_images, train_masks, BATCH_SIZE, HEIGHT, WIDTH,
                                rgb_mask=RGB_MODE, augmentation=data_augmentation, preprocess_fn=preprocess_input)
val_generator = DataGenerator(val_images, val_masks, BATCH_SIZE, HEIGHT, WIDTH,
                              rgb_mask=RGB_MODE, preprocess_fn=preprocess_input)
test_generator = DataGenerator(test_images, test_masks, BATCH_SIZE, HEIGHT, WIDTH,
                               rgb_mask=RGB_MODE, preprocess_fn=preprocess_input)


In [ ]:
model = sm.Unet(
    BACKBONE,
    encoder_weights='imagenet',
    input_shape=IMAGE_SHAPE,
    classes=num_classes,
    activation='softmax',
    encoder_freeze=True,
)

model.compile(
    optimizer='Adam',
    loss=sm.losses.bce_jaccard_loss,
    metrics=[sm.metrics.iou_score, sm.metrics.f1_score],
)

In [ ]:
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=len(val_generator),
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, verbose=1),
        VisualizeProgress(val_generator[0], every_n_epochs=1, num_samples=5, threshold=0.5 if RGB_MODE else None),
    ],
    verbose=1,
)

In [ ]:
model = tf.keras.models.load_model('models/sm-unet-model.h5', compile=False)
model.compile(
    optimizer='Adam',
    loss=sm.losses.bce_jaccard_loss,
    metrics=[sm.metrics.iou_score, sm.metrics.f1_score]
)
model.evaluate(test_generator, steps=len(test_generator))